# Dynamic Pricing — Minimal GRPO Training with Unsloth

**Environment:** Ride-hailing multi-step price negotiation (OpenEnv)

**Model:** Qwen2.5-1.5B-Instruct fine-tuned with GRPO via Unsloth

**What this notebook does:**
1. Installs dependencies (Unsloth + environment)
2. Loads the environment from the HuggingFace Space (or locally)
3. Loads Qwen2.5-1.5B-Instruct in 4-bit via Unsloth
4. Runs a minimal GRPO training loop (Phase 2: platform vs rule-based simulator)
5. Shows before/after completion rate

**Runtime:** GPU required (T4 free tier works). ~30–60 min for 200 steps.

---
**HuggingFace Space:** `https://<your-org>-<space-name>.hf.space`  
**Repo:** `https://huggingface.co/spaces/<your-org>/<space-name>`

## 1. Install Dependencies

In [ ]:
# Install Unsloth (handles torch + bitsandbytes + transformers)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q

# Install environment and other deps
!pip install openenv-core==0.2.3 pydantic>=2.8 fastapi uvicorn requests -q

print("Installation complete.")

In [ ]:
import os, sys, subprocess
from pathlib import Path

# OPTIONAL: clone the HF Space if you don't already have the project in /content.
# Leave empty if you uploaded the folder via the Files panel or mounted Drive.
REPO_URL = ""  # e.g. "https://huggingface.co/spaces/<your-org>/<space-name>"


def _find_project_root():
    """Search common Colab locations for a folder containing ride_hailing_env/."""
    search_roots = [Path("/content"), Path.cwd(), Path.cwd().parent,
                    Path("/content/drive/MyDrive")]
    seen = set()
    for root in search_roots:
        if not root.exists() or root in seen:
            continue
        seen.add(root)
        # direct child match
        for child in root.iterdir():
            if child.is_dir() and (child / "ride_hailing_env" / "__init__.py").exists():
                return child
        # one extra level (covers nested zips that extracted with a wrapper folder)
        for child in root.iterdir():
            if not child.is_dir():
                continue
            try:
                for grand in child.iterdir():
                    if grand.is_dir() and (grand / "ride_hailing_env" / "__init__.py").exists():
                        return grand
            except PermissionError:
                continue
    return None


if REPO_URL and not _find_project_root():
    print(f"Cloning {REPO_URL} ...")
    subprocess.run(["git", "clone", REPO_URL, "/content/dynamic_pricing_env"], check=True)

PROJECT_ROOT = _find_project_root()
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not locate the project root. Either:\n"
        "  1. Set REPO_URL above to your HF Space repo URL, OR\n"
        "  2. Upload the project folder (containing ride_hailing_env/) to /content/, OR\n"
        "  3. Mount Drive and place the folder under MyDrive.\n"
        "Searched: /content, cwd, cwd parent, /content/drive/MyDrive"
    )

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print(f"Project root : {PROJECT_ROOT}")
print(f"sys.path[0]  : {sys.path[0]}")
print(f"cwd          : {os.getcwd()}")

## 2. Verify Environment

In [ ]:
from ride_hailing_env.environment import DynamicPricingEnv

env = DynamicPricingEnv(task_name="easy")
obs = env.reset()

print("Environment reset OK")
print(f"  Trip: {obs.distance_km:.1f} km, {obs.estimated_duration_min:.0f} min")
print(f"  Rider quote: ${obs.rider_quoted_price:.2f}  |  Driver quote: ${obs.driver_quoted_price:.2f}")
print(f"  Max steps: {obs.max_steps}")

# Quick rule-based episode to confirm step() works
result = env.step({"type": "propose_price", "payload": {"price": 15.0}})
print(f"  Step result — reward: {result.reward}  done: {result.done}")
print("\nEnvironment verified OK")

## 3. Load Model with Unsloth (4-bit)

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME   = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LEN  = 1024
LORA_R       = 16

# Load in 4-bit — fits comfortably on T4 (16 GB)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_NAME,
    max_seq_length= MAX_SEQ_LEN,
    load_in_4bit  = True,
    dtype         = None,   # auto-detect
)

# Wrap with LoRA adapter — only q_proj and v_proj are trainable
model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    target_modules = ["q_proj", "v_proj"],
    lora_alpha     = LORA_R,
    lora_dropout   = 0.0,
    bias           = "none",
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}  ({100*trainable/total:.2f}%)")

## 4. Prompt Builder and Price Parser

In [ ]:
import json, re

def build_prompt(obs) -> str:
    """Format the 23-field observation as an instruction prompt."""
    return f"""You are a ride-hailing platform pricing agent. Your goal is to propose a price that both the rider and driver will accept as quickly as possible.

## Current Situation
- Trip: {obs.distance_km:.1f} km, {obs.estimated_duration_min:.0f} min
- Rider quoted price: ${obs.rider_quoted_price:.2f} (their stated max — true max is higher)
- Driver quoted price: ${obs.driver_quoted_price:.2f} (their stated min — true min is lower)
- Rider patience: {obs.rider_patience:.2f} | mood: {obs.rider_mood}
- Driver patience: {obs.driver_patience:.2f} | mood: {obs.driver_mood}
- Step: {obs.step_number + 1}/{obs.max_steps}
- Last rider response: {obs.last_rider_response or 'none'}
- Last driver response: {obs.last_driver_response or 'none'}
- Last proposed price: ${obs.last_proposed_price:.2f if obs.last_proposed_price else 'none'}
- Weather: {obs.weather_condition} | Traffic: {obs.traffic_level} | Surge: {obs.surge_multiplier:.1f}x
- Commission: {obs.commission_rate:.0%} | Operational cost: ${obs.operational_cost:.2f}

## Your Task
Propose a price. Respond with only a JSON object.

{{"price": <number>}}"""


def parse_price(text: str, fallback: float) -> float:
    """Extract price from model output — JSON first, regex fallback."""
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[-1].rsplit("```", 1)[0].strip()
    try:
        return float(json.loads(text)["price"])
    except Exception:
        pass
    nums = re.findall(r"\b\d+\.?\d*\b", text)
    return float(nums[0]) if nums else fallback


print("Prompt builder and parser ready.")

## 5. Rollout Collection

Collect episodes from the environment. Each episode returns a list of
`(prompt, completion, reward)` triples. Only the terminal step is used
for GRPO (outcome-determining generation).

In [ ]:
import torch

def collect_episode(model, tokenizer, env, task="easy"):
    """Run one episode. Returns terminal (prompt, completion, reward)."""
    obs  = env.reset()
    done = False
    terminal = None

    while not done:
        prompt   = build_prompt(obs)
        fallback = (obs.rider_quoted_price + obs.driver_quoted_price) / 2.0
        inputs   = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.inference_mode():
            out = model.generate(
                **inputs,
                max_new_tokens     = 32,
                temperature        = 0.7,
                do_sample          = True,
                pad_token_id       = tokenizer.eos_token_id,
            )
        completion = tokenizer.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        ).strip()

        price  = parse_price(completion, fallback)
        price  = max(1.0, min(500.0, price))
        result = env.step({"type": "propose_price", "payload": {"price": round(price, 2)}})

        terminal = (prompt, completion, result.reward)
        obs  = result.observation
        done = result.done

    return terminal


def collect_batch(model, tokenizer, env, task="easy", batch_size=8):
    """Collect batch_size terminal steps for GRPO."""
    prompts, completions, rewards = [], [], []
    for _ in range(batch_size):
        p, c, r = collect_episode(model, tokenizer, env, task)
        prompts.append(p)
        completions.append(c)
        rewards.append(r)
    return prompts, completions, rewards


print("Rollout functions ready.")

## 6. GRPO Gradient Update

Manual GRPO implementation — group-relative advantage normalisation
over (prompt, completion, reward) tuples collected from the environment.

**Why not TRL's GRPOTrainer?**  
TRL's trainer generates completions internally from a static dataset and
cannot interleave with stateful `env.step()` calls. The math here is
identical to GRPO (Shao et al., 2024).

In [ ]:
from torch.optim import AdamW

def grpo_step(model, tokenizer, optimizer, prompts, completions, rewards,
              num_generations=8, eps=1e-8, max_grad_norm=1.0):
    """One GRPO update step over a collected batch."""
    if not prompts:
        return 0.0

    model.train()
    device    = next(model.parameters()).device
    rewards_t = torch.tensor(rewards, dtype=torch.float32)

    # Group-relative advantage normalisation
    advantages = torch.zeros_like(rewards_t)
    g = max(num_generations, 1)
    for start in range(0, len(rewards_t), g):
        end  = min(start + g, len(rewards_t))
        grp  = rewards_t[start:end]
        mean = grp.mean()
        std  = grp.std() if len(grp) > 1 else torch.tensor(1.0)
        advantages[start:end] = (grp - mean) / (std + eps)
    advantages = advantages.to(device)

    # Policy gradient loss
    total_loss = torch.tensor(0.0, device=device)
    n_valid    = 0

    for prompt, completion, adv in zip(prompts, completions, advantages):
        full_text  = prompt + completion
        enc        = tokenizer(full_text, return_tensors="pt",
                               truncation=True, max_length=512).to(device)
        prompt_enc = tokenizer(prompt, return_tensors="pt",
                               truncation=True, max_length=512).to(device)
        prompt_len = prompt_enc["input_ids"].shape[1]

        if enc["input_ids"].shape[1] <= prompt_len:
            continue

        logits          = model(**enc).logits
        completion_ids  = enc["input_ids"][0, prompt_len:]
        completion_logits = logits[0, prompt_len - 1:-1, :]
        log_probs       = torch.nn.functional.log_softmax(completion_logits, dim=-1)
        token_log_probs = log_probs[
            torch.arange(len(completion_ids), device=device), completion_ids
        ]
        total_loss = total_loss + (-adv * token_log_probs.mean())
        n_valid   += 1

    if n_valid == 0:
        return 0.0

    loss = total_loss / n_valid
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], max_grad_norm
    )
    optimizer.step()
    return loss.item()


optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=5e-6, eps=1e-8
)
print("GRPO optimizer ready.")

## 7. Baseline Evaluation (Before Training)

In [ ]:
TASK       = "easy"
EVAL_EPS   = 20
BATCH_SIZE = 8
NUM_STEPS  = 200   # increase to 500+ for real training

env = DynamicPricingEnv(task_name=TASK)

# Switch to inference mode for eval (2x faster generation)
FastLanguageModel.for_inference(model)

def evaluate(model, tokenizer, env, n=EVAL_EPS):
    rewards, completed = [], 0
    for _ in range(n):
        p, c, r = collect_episode(model, tokenizer, env)
        rewards.append(r)
        if r > 0:
            completed += 1
    avg_r = sum(rewards) / len(rewards) if rewards else 0
    cr    = completed / n
    return avg_r, cr

baseline_reward, baseline_cr = evaluate(model, tokenizer, env)
print(f"Baseline (untrained) — avg_reward: {baseline_reward:.3f}  completion: {baseline_cr:.0%}")

## 8. GRPO Training Loop

In [ ]:
import time

reward_history = []
metrics        = []
step           = 0

print(f"Training — task={TASK}, steps={NUM_STEPS}, batch={BATCH_SIZE}")
print(f"{'─'*65}")
print(f"{'STEP':>6}  {'AVG_R':>7}  {'WIN_AVG':>8}  {'DONE%':>6}  {'LOSS':>8}  {'TIME':>6}")
print(f"{'─'*65}")

while step < NUM_STEPS:
    t0 = time.time()

    # Collect rollouts
    prompts, completions, rewards = collect_batch(
        model, tokenizer, env, TASK, BATCH_SIZE
    )
    reward_history.extend(rewards)

    # GRPO weight update
    loss = grpo_step(model, tokenizer, optimizer,
                     prompts, completions, rewards,
                     num_generations=BATCH_SIZE)

    step    += BATCH_SIZE
    elapsed  = time.time() - t0

    # Monitoring every 50 steps
    if step % 50 < BATCH_SIZE:
        avg_r    = sum(rewards) / len(rewards) if rewards else 0
        window   = reward_history[-100:]
        win_avg  = sum(window) / len(window) if window else 0
        comp_r   = sum(1 for r in rewards if r > 0) / len(rewards) if rewards else 0

        metrics.append({
            "step": step, "avg_reward": round(avg_r, 4),
            "window_avg": round(win_avg, 4),
            "completion_rate": round(comp_r, 3),
            "grpo_loss": round(loss, 6),
        })
        print(f"{step:>6}  {avg_r:>+7.3f}  {win_avg:>+8.3f}  "
              f"{comp_r:>5.0%}  {loss:>8.5f}  {elapsed:>5.1f}s")

print(f"{'─'*65}")
print("Training complete.")

## 8b. Save Training Metrics to Disk

Saves step-level metrics so they persist if Colab disconnects and can be pushed to the repo.

In [ ]:
import os, json as _json

os.makedirs("data", exist_ok=True)

metrics_payload = {
    "task": TASK,
    "num_steps": NUM_STEPS,
    "batch_size": BATCH_SIZE,
    "baseline": {"avg_reward": round(baseline_reward, 4), "completion_rate": round(baseline_cr, 3)},
    "steps": metrics,
}

METRICS_PATH = f"data/training_metrics_platform_{TASK}.json"
with open(METRICS_PATH, "w") as _f:
    _json.dump(metrics_payload, _f, indent=2)

print(f"Metrics saved → {METRICS_PATH}  ({len(metrics)} checkpoints)")


## 9. Post-Training Evaluation

In [ ]:
# Switch back to inference mode for fast eval
FastLanguageModel.for_inference(model)

post_reward, post_cr = evaluate(model, tokenizer, env)

print("\n" + "="*50)
print("  RESULTS")
print("="*50)
print(f"  {'':25} {'REWARD':>8}  {'DEAL%':>6}")
print(f"  {'─'*42}")
print(f"  {'Baseline (untrained)':25} {baseline_reward:>+8.3f}  {baseline_cr:>5.0%}")
print(f"  {'After GRPO training':25} {post_reward:>+8.3f}  {post_cr:>5.0%}")
print(f"  {'Δ improvement':25} {post_reward - baseline_reward:>+8.3f}  {post_cr - baseline_cr:>+5.0%}")
print("="*50)

## 9b. Save Evaluation Results to Disk

Saves the before/after numbers to JSON so they can be pushed to the repo for judges.

In [ ]:
eval_payload = {
    "task": TASK,
    "model": "Qwen2.5-1.5B-Instruct + LoRA r=16",
    "stages": [
        {
            "stage": "A",
            "label": "Untrained vs rule-based",
            "avg_reward": round(baseline_reward, 4),
            "completion_rate": round(baseline_cr, 3),
        },
        {
            "stage": "B",
            "label": "Platform_v0 vs rule-based (after GRPO)",
            "avg_reward": round(post_reward, 4),
            "completion_rate": round(post_cr, 3),
        },
    ],
    "delta_reward": round(post_reward - baseline_reward, 4),
    "delta_completion": round(post_cr - baseline_cr, 3),
}

EVAL_PATH = "data/colab_evaluation.json"
with open(EVAL_PATH, "w") as _f:
    _json.dump(eval_payload, _f, indent=2)

print(f"Evaluation saved → {EVAL_PATH}")
print()
print(f"  Stage A (untrained):  reward={baseline_reward:+.3f}  completion={baseline_cr:.0%}")
print(f"  Stage B (trained):    reward={post_reward:+.3f}  completion={post_cr:.0%}")
print(f"  Delta:                reward={post_reward-baseline_reward:+.3f}  completion={post_cr-baseline_cr:+.0%}")


## 10. Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

if metrics:
    steps   = [m["step"] for m in metrics]
    avg_rs  = [m["avg_reward"] for m in metrics]
    win_avgs= [m["window_avg"] for m in metrics]
    comp_rs = [m["completion_rate"] * 100 for m in metrics]
    losses  = [m["grpo_loss"] for m in metrics]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f"GRPO Training — {TASK} task", fontsize=13, fontweight="bold")

    axes[0].plot(steps, avg_rs, alpha=0.4, color="#4C72B0", label="batch avg")
    axes[0].plot(steps, win_avgs, linewidth=2, color="#4C72B0", label="100-step window")
    axes[0].axhline(baseline_reward, color="gray", linestyle="--", label=f"baseline {baseline_reward:.2f}")
    axes[0].set_title("Reward")
    axes[0].set_xlabel("Step")
    axes[0].legend(fontsize=8)
    axes[0].grid(alpha=0.3)

    axes[1].plot(steps, comp_rs, linewidth=2, color="#55A868")
    axes[1].axhline(baseline_cr * 100, color="gray", linestyle="--",
                    label=f"baseline {baseline_cr:.0%}")
    axes[1].axhline(70, color="#55A868", linestyle=":", label="target 70%")
    axes[1].set_title("Completion Rate (%)")
    axes[1].set_xlabel("Step")
    axes[1].set_ylim(0, 105)
    axes[1].legend(fontsize=8)
    axes[1].grid(alpha=0.3)

    axes[2].plot(steps, losses, linewidth=2, color="#C44E52")
    axes[2].set_title("GRPO Loss")
    axes[2].set_xlabel("Step")
    axes[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig("data/training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: data/training_curves.png")

## 11. Save LoRA Checkpoint

In [ ]:
SAVE_PATH = "/content/platform_lora_colab"

# Save adapter only — never upcast 4-bit + naive merge (degrades quality)
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"LoRA adapter saved to {SAVE_PATH}")

# Optional: push to HuggingFace Hub
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_folder(
#     folder_path=SAVE_PATH,
#     repo_id="<your-org>/dynamic-pricing-checkpoints",
#     repo_type="model",
#     path_in_repo="checkpoints/phase2/platform_lora",
# )
# print("Uploaded to HF Hub.")

## 12. Run a Demo Episode with the Trained Model

In [ ]:
FastLanguageModel.for_inference(model)
env  = DynamicPricingEnv(task_name=TASK)
obs  = env.reset()
hidden = env.get_hidden_state()
done = False

print(f"\nDEMO EPISODE — trained model, task={TASK}")
print(f"Rider ceiling: ${hidden.rider_max_willingness:.2f}  "
      f"Driver floor: ${hidden.driver_min_willingness:.2f}")
print()

while not done:
    prompt   = build_prompt(obs)
    fallback = (obs.rider_quoted_price + obs.driver_quoted_price) / 2.0
    inputs   = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        out = model.generate(
            **inputs, max_new_tokens=32, temperature=0.7,
            do_sample=True, pad_token_id=tokenizer.eos_token_id
        )
    completion = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

    price  = max(1.0, min(500.0, parse_price(completion, fallback)))
    result = env.step({"type": "propose_price", "payload": {"price": round(price, 2)}})

    r_accept = price <= hidden.rider_max_willingness
    d_accept = price >= hidden.driver_min_willingness
    print(f"  Step {obs.step_number + 1}/{obs.max_steps}  "
          f"price=${price:.2f}  "
          f"rider={'✓' if r_accept else '✗'}  "
          f"driver={'✓' if d_accept else '✗'}")

    done = result.done
    obs  = result.observation

outcome = result.info.get("outcome", {})
if outcome.get("ride_completed"):
    print(f"\n  ✅ DEAL CLOSED — reward: {result.reward:.3f}")
elif outcome.get("timed_out"):
    print(f"\n  ⏰ TIMED OUT — reward: {result.reward:.3f}")
else:
    print(f"\n  ❌ CANCELLED — reward: {result.reward:.3f}")